[![Roboflow Notebooks](https://media.roboflow.com/notebooks/template/bannertest2-2.png?ik-sdk-version=javascript-1.4.3&updatedAt=1672932710194)](https://github.com/roboflow/notebooks)

# How to Train a Re-ID Model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/re-ID/blob/main/docs/cookbooks/how-to-train-reid-model.ipynb)

A Re-ID model finds the same person across images and cameras by how close their embeddings are. To do it accurately, the model has to be trained for your domain. In this notebook you'll learn how to fine-tune a pretrained Re-ID model on Market-1501, keep the epoch that scores best on people held out for validation, and measure how much it improves on the test set. The steps are the same for any dataset of crops with known identities.

> **Runtime:** T4 GPU.

## Setup

### Check GPU availability

Training runs on every crop many times, so use a GPU runtime. If the cell below fails, go to `Runtime` → `Change runtime type` and pick a GPU.

In [ ]:
!nvidia-smi

### Install `reid`

In [ ]:
!pip install -q "reid @ git+https://github.com/roboflow/re-ID@feat/training-plots"

### Imports

In [ ]:
import os
import zipfile
from pathlib import Path

import gdown
import numpy as np
from IPython.display import Image, display

from reid import ReIDModel, RetrievalProtocol, load_market1501_test, load_market1501_train, plot_training_history

## Download Market-1501

Market-1501 is a public research benchmark of people seen by 6 cameras in front of a supermarket. Its training split holds 12,936 images of 751 people. Its test split holds 3,368 query images and a gallery of 19,732 images of 750 other people, false detections included. We download the archive the [dataset authors](https://zheng-lab-anu.github.io/Project/project_reid.html) link to, 153 MB. Review the dataset's terms before use.

In [ ]:
MARKET_ZIP = "/content/Market-1501.zip"
MARKET_DIR = "/content/Market-1501-v15.09.15"

if not os.path.exists(MARKET_DIR):
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=MARKET_ZIP, quiet=False)
    with zipfile.ZipFile(MARKET_ZIP) as archive:
        archive.extractall("/content")

## Hold out people for validation

To pick the best epoch we need a validation set, and Market-1501 has none. To make one, we use `train_val_split`, which moves every image of 100 people from the training split to a validation set, so `train` scores the model on people it never saw.

In [ ]:
market_train = load_market1501_train(MARKET_DIR)
train_crops, val_crops = market_train.train_val_split(num_val_identities=100, seed=42)
query, gallery = load_market1501_test(MARKET_DIR)

print(f"train: {len(train_crops)} images of {len(np.unique(train_crops.pids))} people")
print(f"val: {len(val_crops)} images of {len(np.unique(val_crops.pids))} people")
print(f"test: {len(query)} query images, {len(gallery)} gallery images")

## Score the model before training

We start from `osnet_x1_0_msmt17_combineall`, a general model pretrained on pedestrians from MSMT17, and score it on the test split. For that we use `RetrievalProtocol`, which ranks the gallery for each query by how close their embeddings are. `rank1` is the percentage of queries whose closest gallery image shows the same person, and `map` averages, over the queries, the precision at every rank where a match appears.

In [ ]:
model = ReIDModel.from_pretrained("osnet_x1_0_msmt17_combineall")
protocol = RetrievalProtocol()

before = model.evaluate(query, gallery, protocol=protocol, verbose=False)
print(f"rank1 {before.metrics['rank1']:.1f}  map {before.metrics['map']:.1f}")

## Train

We train with `train`'s default settings, which suit a pretrained model. To score the model on the held-out people before training and after every epoch, we pass them as `val_data`, and `train` saves the epoch with the highest mAP to `best/` in `output_dir`. After every epoch, `train` also plots the embeddings of 10 validation people, under that epoch's line. To load and augment the crops in 2 background processes, we set `num_workers=2`.

In [ ]:
history = model.train(train_crops, output_dir="runs/market1501", val_data=val_crops, num_workers=2)

## Plot the training curves

To plot the losses, the validation scores and the learning rate by epoch, we use `plot_training_history`.

In [ ]:
fig = plot_training_history("runs/market1501")

Validation mAP rises fast once the backbone starts training and then slowly, with the last gains after the learning-rate drops. It is still at its best near the end of the run, marked by the dashed line, so the model hasn't started overfitting the training people. With only 100 validation people, Rank-1 stays close to 100 the whole run, so mAP is the score that shows the progress.

The run folder holds `last/`, the model after the last epoch, and `best/`, the epoch with the highest mAP on the held-out people. Both load with `from_pretrained` like any model. Next to them, `metrics.csv` holds the rows of `history`, `training_config.json` the run's settings, and `embeddings/` one plot per epoch.

In [ ]:
print(sorted(path.name for path in Path("runs/market1501").iterdir()))

## Watch the embeddings separate

To see how the embeddings changed, we compare the plot before training with the one after the last epoch.

In [ ]:
display(Image(filename="runs/market1501/embeddings/before_training.png"))
display(Image(filename="runs/market1501/embeddings/epoch_40.png"))

Before training, the points of different people mix in shared clouds. After the last epoch, each person's points sit closer together and most people stand apart, though a few still overlap. The plot keeps only the 2 directions along which the 512-dimensional embeddings vary most, so people who overlap in it can still be apart in the full embedding.

## Score the model after training

We now compare the last epoch, which `model` holds, and the best epoch (loaded from `best/`), on the test set. Neither training nor validation saw these people.

In [ ]:
best_model = ReIDModel.from_pretrained("runs/market1501/best")

results = {
    "before training": before,
    "last epoch": model.evaluate(query, gallery, protocol=protocol, verbose=False),
    "best epoch": best_model.evaluate(query, gallery, protocol=protocol, verbose=False),
}
print(f"{'':<18}{'rank1':>8}{'map':>8}")
for name, result in results.items():
    print(f"{name:<18}{result.metrics['rank1']:>8.1f}{result.metrics['map']:>8.1f}")

Fine-tuning raises rank-1 on the test set from 66.5 to 93.8 and mAP from 40.0 to 85.1. The best and the last epoch score about the same, because by the end of the run the learning rate is so low that the weights barely move.

You just fine-tuned a Re-ID model on Market-1501 and kept its best epoch. Nice work! For more information see our [documentation guides](https://reid.roboflow.com/guides/training/), which cover more details!

Got feedback or ideas? Open an issue on [GitHub Issues](https://github.com/roboflow/re-ID/issues).